## Installation des dépendances

In [30]:
!pip install openmeteo_requests
!pip install requests_cache
!pip install retry_requests
!pip install openpyxl

## Fonctions d'appel de l'API

In [39]:
import openmeteo_requests
import requests_cache
import pandas as pd
from retry_requests import retry
import time

# ── Setup session avec cache 1h et retry automatique ──────────────
cache_session = requests_cache.CachedSession('.cache_meteo', expire_after=3600)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

# ── Variables météo à récupérer ───────────────────────────────────
VARIABLES_HORAIRES = [
    "temperature_2m",          # Température air (°C)
    "relative_humidity_2m",     # Humidité relative (%)
    "apparent_temperature",     # Température ressentie (°C)
    "precipitation",            # Précipitations (mm)
    "wind_speed_10m",           # Vitesse vent à 10m (km/h)
    "wind_gusts_10m",           # Rafales (km/h)
    "wind_direction_10m",       # Direction vent (°)
    "weather_code",             # Code météo WMO
    "cloud_cover",              # Couverture nuageuse (%)
    "pressure_msl",             # Pression mer (hPa)
    "shortwave_radiation",      # Rayonnement solaire (W/m²)
    "visibility",               # Visibilité (m) — important côtier
]

VARIABLES_JOURNALIERES = [
    "temperature_2m_max",
    "temperature_2m_min",
    "precipitation_sum",
    "wind_speed_10m_max",
    "wind_gusts_10m_max",
    "wind_direction_10m_dominant",
    "sunrise",
    "sunset",
    "uv_index_max",
    "weather_code",
]

# ── Récupération des données ──────────────────────────────────────
def fetch_meteo_departement(nom, lat, lon, facade):
    params = {
        "latitude":             lat,
        "longitude":            lon,
        "hourly":               VARIABLES_HORAIRES,
        "daily":                VARIABLES_JOURNALIERES,
        "current": ["temperature_2m", "wind_speed_10m",
                     "wind_gusts_10m", "precipitation",
                     "weather_code", "pressure_msl"],
        "models":               "meteofrance_seamless",
        "timezone":             "Europe/Paris",
        "forecast_days":        4,
        "wind_speed_unit":      "kmh",
        "temperature_unit":     "celsius",
        "precipitation_unit":   "mm",
        "temporal_resolution":  "hourly_2",  # ← 1 valeur toutes les 2h
    }
    url = "https://api.open-meteo.com/v1/meteofrance"
    responses = openmeteo.weather_api(url, params=params)
    r = responses[0]

    h = r.Hourly()
    times = pd.date_range(
        start=pd.to_datetime(h.Time(), unit="s", utc=True),
        end=pd.to_datetime(h.TimeEnd(), unit="s", utc=True),
        freq=pd.Timedelta(seconds=h.Interval()),
        inclusive="left"
    ).tz_convert("Europe/Paris")

    df_h = pd.DataFrame({"time": times})
    for i, var in enumerate(VARIABLES_HORAIRES):
        df_h[var] = h.Variables(i).ValuesAsNumpy()
    df_h["departement"] = nom
    df_h["facade"]      = facade
    df_h["latitude"]    = lat
    df_h["longitude"]   = lon

    return df_h

## Construction de la base de donnée nautique

In [32]:
data = pd.read_csv("https://minio.lab.sspcloud.fr/matteo/data-es.csv", sep=";")
data_nautique = data[data['equip_type_famille'] == "Site d'activités aquatiques et nautiques"]


colonnes = ['equip_numero', "inst_numero", "inst_nom", "inst_adresse", "inst_cp", "dep_code", "reg_code", "dep_nom", "reg_nom", "lib_bdv", "equip_nom", "equip_type_name", "equip_coordonnees", "aps_name"]
data_nautique = data_nautique[colonnes].reset_index(drop=True).copy()
mots = ["surf", "ski", "voile", "foil", "wake"]
pattern = "|".join(mots)
data_nautique = data_nautique[data_nautique['aps_name'].str.contains(pattern, case=False, na=False)]
data_nautique['dep_code'] = data_nautique['dep_code'].astype(str)
data_nautique

/tmp/ipykernel_38633/3108644381.py:1: DtypeWarning: Columns (6,8,19,23,25,29) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv("https://minio.lab.sspcloud.fr/matteo/data-es.csv", sep=";")


,equip_numero,inst_numero,inst_nom,inst_adresse,inst_cp,dep_code,reg_code,dep_nom,reg_nom,lib_bdv,equip_nom,equip_type_name,equip_coordonnees,aps_name
4,E003I974130041,I974130041,Baie de St Leu,rue de la Campagnie des Indes,97416,974,4.0,La Réunion,La Réunion,Saint-Leu,La Cafrine,Site d'activités aquatiques et nautiques,"-21.1867, 55.2866",Surf
5,E003I974130046,I974130046,Cimetière de St Leu,RN 1 Grand Fond,97416,974,4.0,La Réunion,La Réunion,Saint-Leu,Le cimetière,Site d'activités aquatiques et nautiques,"-21.1858, 55.287",Surf
8,E001I690640001,I690640001,BASE DE LOISIRS DE CONDRIEU,LA PRESQU'ILE,69420,69,84.0,Rhône,Auvergne-Rhône-Alpes,Condrieu,Base de loisirs-plan d'eau,Site d'activités aquatiques et nautiques,"45.45552, 4.77446","Canoë de randonnée,Téléski nautique"
15,E004I011840003,I011840003,CHAMBOD - CENTRE UDPA,NaN,1250,1,84.0,Ain,Auvergne-Rhône-Alpes,Ceyzériat,SITE MIXTE,Site d'activités aquatiques et nautiques,"46.12417, 5.42917","Baignade loisirs,Canoë de randonnée,Kayak-polo..."
17,E005I412690005,I412690005,COMPLEXE SPORTIF DES GRANDS PRES,Rue Geoffroy Martel,41100,41,24.0,Loir-et-Cher,Centre-Val de Loire,Vendôme,Base nautique des Grands Prés,Site d'activités aquatiques et nautiques,"47.79099, 1.0741","Canoë de randonnée,Planche à Voile"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11570,E006I581800003,I581800003,Base Nautique des Settons,Lac des Settons D 193 Montsauche les settons,58230,58,27.0,Nièvre,Bourgogne-Franche-Comté,Saulieu,Base de plein air multiactivités,Site d'activités aquatiques et nautiques,"47.18739, 4.0643","Aviron,Canoë de randonnée,Dériveur / Multicoqu..."
11576,E006I721820001,I721820001,Espace touristique du lac de Mansigné,Route de Plessis,72510,72,52.0,Sarthe,Pays de la Loire,Écommoy,Plan d'eau,Site de pêche,"47.75788, 0.13017",Dériveur / Multicoques / Courses océaniques / ...
11583,E004I880820003,I880820003,Pôle sports nature,La grande haie,88110,88,44.0,Vosges,Grand Est,Raon-l'Étape,Site d'activités aquatiques,Site d'activités aquatiques et nautiques,"48.4565, 6.95106","Baignade loisirs,Canoë de randonnée,Planche à ..."
11585,E004I890240043,I890240043,Base nautique de VAUX,22 Rue du poiry Vaux,89000,89,27.0,Yonne,Bourgogne-Franche-Comté,Auxerre,Base de ski nautique - Rte de Poiry,Stade de ski nautique,"47.76078, 3.59631",Ski nautique classique / Course / à figures li...


## Filtre le dataframe suivant la ville choisie et le rayon autour de celle ci

In [38]:
from math import radians, sin, cos, sqrt, atan2

# ── Fonction distance Haversine ───────────────────────────────────
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Rayon de la Terre en km
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))

# ── Filtre par ville et rayon ─────────────────────────────────────
def filter_by_ville(df, nom_ville, rayon_km):
    """
    df        : DataFrame source contenant equip_coordonnees et lib_bdv
    nom_ville : Nom de la ville (colonne lib_bdv)
    rayon_km  : Rayon de recherche en kilomètres
    """
    # Récupère les coordonnées de la ville
    ville_rows = df[df["lib_bdv"].str.lower() == nom_ville.lower()]
    
    if ville_rows.empty:
        print(f"⚠️  Ville '{nom_ville}' introuvable dans le dataframe.")
        print("Villes disponibles :", sorted(df["lib_bdv"].dropna().unique().tolist()))
        return None

    lat_ville = ville_rows["latitude"].iloc[0]
    lon_ville = ville_rows["longitude"].iloc[0]
    print(f"✓ Ville trouvée : {nom_ville} ({lat_ville}, {lon_ville})")

    # Calcule la distance de chaque équipement par rapport à la ville
    df = df.copy()
    df["distance_km"] = df.apply(
        lambda row: haversine(lat_ville, lon_ville, row["latitude"], row["longitude"]),
        axis=1
    )

    # Filtre par rayon
    df_filtre = df[df["distance_km"] <= rayon_km].copy()
    df_filtre = df_filtre.sort_values("distance_km").reset_index(drop=True)

    print(f"✓ {len(df_filtre)} équipements dans un rayon de {rayon_km} km autour de {nom_ville}")
    return df_filtre

df_source = data_nautique.copy()

# ── Utilisation ───────────────────────────────────────────────────
# Assure-toi que les colonnes latitude/longitude sont bien présentes
df_source[["latitude", "longitude"]] = (
    df_source["equip_coordonnees"]
    .str.split(",", expand=True)
    .astype(float)
)

# Exemple : équipements dans un rayon de 50 km autour de Brest
df_filtre = filter_by_ville(df_source, nom_ville="Brest", rayon_km=20)

# Aperçu du résultat
if df_filtre is not None:
    print(df_filtre[["equip_nom", "lib_bdv", "dep_nom", "distance_km"]].head(10).to_string(index=False))

✓ Ville trouvée : Brest (48.36176, -4.77654)
✓ 37 équipements dans un rayon de 20 km autour de Brest
                                              equip_nom     lib_bdv   dep_nom  distance_km
                                        Port du Conquet       Brest Finistère     0.000000
               Site de surf, kite surf, planche à voile       Brest Finistère     0.843934
       Zone de mouillages collectifs de l'Anse d'Illien Saint-Renan Finistère     2.428489
Zone de Mouillages collectifs de Kerhornou - Porsmoguer Saint-Renan Finistère     4.728881
                                    Point Passion Plage       Brest Finistère     5.615234
                       Centre Nautique NPI Plougonvelin       Brest Finistère     5.615332
     Anse de Bertheaume - zone de mouillages collectifs       Brest Finistère     6.205048
                          Centre Nautique NPI Plouarzel Saint-Renan Finistère     8.945464
                                       Port de Porspaul Saint-Renan Finistère   

## Récupération des données météo avec l'API

In [ ]:
# ── Boucle de récupération météo ──────────────────────────────────
all_dfs = []
for _, row in df_filtre.iterrows():
    nom    = row["dep_nom"]
    lat    = row["latitude"]
    lon    = row["longitude"]
    facade = "Inconnue"

    print(f"Récupération : {nom}...")
    df = fetch_meteo_departement(nom, lat, lon, facade)

    # Propagation des colonnes utiles
    df["dep_nom"]         = row["dep_nom"]
    df["dep_code"]        = row["dep_code"]
    df["reg_nom"]         = row["reg_nom"]
    df["equip_numero"]    = row["equip_numero"]
    df["equip_nom"]       = row["equip_nom"]
    df["equip_type"]      = row["equip_type_name"]
    df["inst_nom"]        = row["inst_nom"]
    df["inst_adresse"]    = row["inst_adresse"]
    df["aps_name"]        = row["aps_name"]

    all_dfs.append(df)
    time.sleep(0.1)

df_cotes = pd.concat(all_dfs, ignore_index=True)
print(f"\n✓ {len(df_cotes)} lignes — {df_cotes['dep_nom'].nunique()} départements")

Récupération : Finistère...
Récupération : Finistère...
Récupération : Finistère...
Récupération : Finistère...
Récupération : Finistère...
Récupération : Finistère...
Récupération : Finistère...
Récupération : Finistère...
Récupération : Finistère...
Récupération : Finistère...
Récupération : Finistère...
Récupération : Finistère...
Récupération : Finistère...
Récupération : Finistère...
Récupération : Finistère...
Récupération : Finistère...
Récupération : Finistère...
Récupération : Finistère...
Récupération : Finistère...
Récupération : Finistère...
Récupération : Finistère...
Récupération : Finistère...
Récupération : Finistère...
Récupération : Finistère...
Récupération : Finistère...
Récupération : Finistère...
Récupération : Finistère...
Récupération : Finistère...
Récupération : Finistère...
Récupération : Finistère...
Récupération : Finistère...
Récupération : Finistère...
Récupération : Finistère...
Récupération : Finistère...
Récupération : Finistère...
Récupération : Finis

## Exportation des données méteo

In [35]:
# ── Export ───────────────────────────────────────────────────────

# Supprime la timezone pour compatibilité Excel
df_export = df_cotes.copy()
df_export["time"] = df_export["time"].dt.tz_localize(None)

# ── CSV ──────────────────────────────────────────────────────────
df_export.to_csv("meteo_cotes_france.csv", index=False, encoding="utf-8-sig")
print("✓ CSV exporté : meteo_cotes_france.csv")

# ── Conditions actuelles par département ─────────────────────────
now_df = df_export[df_export["time"] == df_export.groupby("dep_nom")["time"].transform("min")]
print("\n── Conditions actuelles ──")
print(now_df[["dep_nom", "equip_nom", "temperature_2m",
              "wind_speed_10m", "wind_gusts_10m", "precipitation"]].to_string(index=False))

# ── Vent max par département ──────────────────────────────────────
vent_max = (df_export
    .groupby("dep_nom")["wind_gusts_10m"]
    .max()
    .sort_values(ascending=False)
    .reset_index())
vent_max.columns = ["Département", "Rafales max (km/h)"]
print("\n── Rafales max par département ──")
print(vent_max.to_string(index=False))

# ── Alerte pluie : équipements > 5mm sur 4 jours ─────────────────
pluie_alert = (df_export
    .groupby(["dep_nom", "equip_nom"])["precipitation"]
    .sum()
    .reset_index()
    .query("precipitation > 5")
    .sort_values("precipitation", ascending=False))
pluie_alert.columns = ["Département", "Équipement", "Précipitations totales (mm)"]
print("\n── Équipements avec pluie > 5mm ──")
print(pluie_alert.to_string(index=False))

# ── Export Excel avec une feuille par département ─────────────────
with pd.ExcelWriter("meteo_cotes_france.xlsx", engine="openpyxl") as writer:
    for dep, group in df_export.groupby("dep_nom"):
        sheet_name = dep[:31]  # Excel limite les noms d'onglets à 31 caractères
        group.to_excel(writer, sheet_name=sheet_name, index=False)
print("\n✓ Excel exporté : meteo_cotes_france.xlsx")

✓ CSV exporté : meteo_cotes_france.csv

── Conditions actuelles ──
  dep_nom                                                      equip_nom  temperature_2m  wind_speed_10m  wind_gusts_10m  precipitation
Finistère                                                Port du Conquet       12.745500       27.162708       41.039997            0.0
Finistère                       Site de surf, kite surf, planche à voile       12.454000       29.964457       40.320000            0.0
Finistère               Zone de mouillages collectifs de l'Anse d'Illien       12.141000       29.964457       37.799999            0.0
Finistère        Zone de Mouillages collectifs de Kerhornou - Porsmoguer       12.356000       26.867676       39.599998            0.0
Finistère                                            Point Passion Plage       12.237000       23.904108       37.799999            0.0
Finistère                               Centre Nautique NPI Plougonvelin       12.237000       23.904108       37.799

In [36]:
df = pd.read_csv("meteo_cotes_france.csv")
df


,time,temperature_2m,relative_humidity_2m,apparent_temperature,precipitation,wind_speed_10m,wind_gusts_10m,wind_direction_10m,weather_code,cloud_cover,...,longitude,dep_nom,dep_code,reg_nom,equip_numero,equip_nom,equip_type,inst_nom,inst_adresse,aps_name
0,2026-04-16 00:00:00,12.745500,89.0,9.225182,0.0,27.162708,41.039997,214.743240,0.0,5.0,...,-4.77654,Finistère,29.0,Bretagne,E001I290400005,Port du Conquet,Zone de mouillage,Port du Conquet,Port du conquet,Dériveur / Multicoques / Courses océaniques / ...
1,2026-04-16 01:00:00,12.695499,90.0,8.935804,0.0,29.598919,45.360000,221.054720,2.0,71.0,...,-4.77654,Finistère,29.0,Bretagne,E001I290400005,Port du Conquet,Zone de mouillage,Port du Conquet,Port du conquet,Dériveur / Multicoques / Courses océaniques / ...
2,2026-04-16 02:00:00,12.545500,91.0,8.882533,0.0,28.551426,47.519997,228.066560,3.0,100.0,...,-4.77654,Finistère,29.0,Bretagne,E001I290400005,Port du Conquet,Zone de mouillage,Port du Conquet,Port du conquet,Dériveur / Multicoques / Courses océaniques / ...
3,2026-04-16 03:00:00,12.495500,90.0,9.072404,0.0,26.739542,42.480000,223.363460,3.0,100.0,...,-4.77654,Finistère,29.0,Bretagne,E001I290400005,Port du Conquet,Zone de mouillage,Port du Conquet,Port du conquet,Dériveur / Multicoques / Courses océaniques / ...
4,2026-04-16 04:00:00,12.495500,91.0,9.169737,0.0,26.230639,41.760002,223.331700,3.0,100.0,...,-4.77654,Finistère,29.0,Bretagne,E001I290400005,Port du Conquet,Zone de mouillage,Port du Conquet,Port du conquet,Dériveur / Multicoques / Courses océaniques / ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13243,2026-04-19 19:00:00,13.014501,64.0,8.292374,0.0,26.089443,45.719997,50.599370,2.0,58.0,...,-4.27947,Finistère,29.0,Bretagne,E001I292870005,Kite sans frontière,Site d'activités aquatiques et nautiques,Plage de Keremma,la sabliere,Cerf volant de traction (glisse aérotractée) e...
13244,2026-04-19 20:00:00,12.264501,67.0,7.702909,0.0,24.912935,44.639996,52.633263,2.0,73.0,...,-4.27947,Finistère,29.0,Bretagne,E001I292870005,Kite sans frontière,Site d'activités aquatiques et nautiques,Plage de Keremma,la sabliere,Cerf volant de traction (glisse aérotractée) e...
13245,2026-04-19 21:00:00,11.364500,70.0,7.107039,0.0,22.476227,42.120000,58.091934,3.0,99.0,...,-4.27947,Finistère,29.0,Bretagne,E001I292870005,Kite sans frontière,Site d'activités aquatiques et nautiques,Plage de Keremma,la sabliere,Cerf volant de traction (glisse aérotractée) e...
13246,2026-04-19 22:00:00,10.814501,71.0,6.682627,0.0,21.120682,37.440000,60.376347,3.0,97.0,...,-4.27947,Finistère,29.0,Bretagne,E001I292870005,Kite sans frontière,Site d'activités aquatiques et nautiques,Plage de Keremma,la sabliere,Cerf volant de traction (glisse aérotractée) e...
